In [158]:
import re
import pandas as pd

In [159]:
# load the dataset
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [160]:
# find the length of positive and negative reviews
def number_reviews(df:pd.DataFrame) -> tuple[float]:
    """
    This function finds the length of positive and negative review
    """
    positive_reviews_length = df[df["sentiment"] == "positive"].shape[0]
    negative_reviews_length = df[df["sentiment"] == "negative"].shape[0]
    return positive_reviews_length, negative_reviews_length

number_reviews(df=df)

(25000, 25000)

In [161]:
positive_words = ["wonderful","masterpiece", "brillant"]
negative_words = ["boring", "dreadful", "disgrateful", "painful"]

In [162]:
def contain_keyword(text:str, keyword:str) -> bool:
    """this is an helper function to handle reviews"""
    return bool(re.search(rf"\b{re.escape(keyword)}\b", text))


In [163]:
def number_positive_review(df:pd.DataFrame, keyword:str, sentiment_type:str) -> int:
    """
    Helper function to find the number of time the word is in positive review
    """
    number_positive = []
    for reviews in df[df["sentiment"] == sentiment_type]["review"]:
        if contain_keyword(text=reviews, keyword=keyword):
            number_positive.append(contain_keyword(text=reviews, keyword=keyword))

    return len(number_positive)

In [164]:
def probability_pos_keyword(number_positive:float):
    """
    This helper function finds the probability of finding a keyword in a positive review
    """
    probability_pos = (number_positive/number_reviews(df=df)[0])
    return probability_pos

probability_pos_keyword(number_positive=number_positive_review(df=df, keyword=positive_words[0], sentiment_type="positive"))

0.08532

In [165]:
def probability_not_finding_pos(positive_probability:float) -> float:
    """
    helper function to get the probability of not finding the keyword in a positive key
    """
    probability_neg = 1 - positive_probability
    return probability_neg

probability_not_finding_pos(positive_probability=probability_pos_keyword(number_positive=number_positive_review(df=df, keyword=positive_words[0], sentiment_type="positive")))

0.91468

In [166]:
def total_probability_dataset(df:pd.DataFrame, keyword:str) -> float:
    """helper function to get the total probability inside the dataset"""
    found = []
    for reviews in df["review"]:
        if contain_keyword(text=reviews, keyword=keyword):
            found.append(contain_keyword(text=reviews, keyword=keyword))
    probability = len(found)/len(df)
    return probability

### get now the probability of a review being negative given that it contains wonderful

In [167]:
probabiltity_neg = (number_reviews(df=df)[1]/len(df))*100
print(f"probability of finding wonderful {probabiltity_neg:.2f}%")

probability of finding wonderful 50.00%


In [168]:
def bayesian_probability_positive(df: pd.DataFrame, keyword: str) -> float:
    """
    Computes P(positive | keyword) via Bayes' theorem, reusing the existing helpers
    """
    count_pos = number_positive_review(df=df, keyword=keyword, sentiment_type="positive")
    count_neg = number_positive_review(df=df, keyword=keyword, sentiment_type="negative")

    p_keyword_given_positive = count_pos / number_reviews(df=df)[0]
    p_positive = number_reviews(df=df)[0] / len(df)
    p_keyword = (count_pos + count_neg) / len(df)

    return (p_keyword_given_positive * p_positive) / p_keyword

for keyword in positive_words:
    probability = bayesian_probability_positive(df=df, keyword=keyword)
    print(f"P(positive | '{keyword}') = {probability:.3f} ({probability*100:.2f}%)")

P(positive | 'wonderful') = 0.812 (81.23%)
P(positive | 'masterpiece') = 0.729 (72.86%)
P(positive | 'brillant') = 0.800 (80.00%)
